# Pipeline de ETL: Global Video Game Sales
**Objetivo:** Extração, limpeza e transformação da base de dados bruta para consumo em dashboards de Business Intelligence.

In [23]:
# 1. Importações
import pandas as pd

In [24]:
# 2. Variáveis Globais e Caminhos
caminho_original = '../data/base_original.csv'
caminho_tratada = '../data/base_tratada.csv'

In [25]:
# 3. Extração / Leitura
df = pd.read_csv(caminho_original)

In [ ]:
# 4. Inspeção Inicial
display(df.head())
display(df.info())

### 2. Descarte de Features
Remoção de colunas sem valor analítico para o escopo do projeto (links de imagens e metadados de atualização). Esta etapa reduz a dimensionalidade do dataset e otimiza o consumo de memória na ingestão de dados.

In [27]:
colunas_para_remover = ['img', 'last_update']
colunas_presentes = [col for col in colunas_para_remover if col in df.columns]

df = df.drop(columns=colunas_presentes)

### 3. Conversão de Tipos (Type Casting)
Adequação dos tipos de dados para numéricos e temporais (`datetime`). A coerção de erros (`errors='coerce'`) garante que anomalias na formatação original sejam padronizadas como dados nulos (NaN/NaT) para facilitar o tratamento subsequente.

In [28]:
if 'release_date' in df.columns:
    df['release_date'] = pd.to_datetime(df['release_date'], dayfirst=True, errors='coerce')

colunas_numericas = [
    'critic_score', 
    'total_sales', 
    'na_sales', 
    'jp_sales', 
    'pal_sales', 
    'other_sales'
]

for col in colunas_numericas:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

### 4. Normalização de Strings
Aplicação de formatação padrão (Title Case) e remoção de espaços nas extremidades (strip) nas variáveis categóricas. Essa etapa previne a fragmentação de grupos e categorias duplicadas nas agregações do dashboard.

In [29]:
colunas_texto = ['genre', 'console', 'publisher', 'developer']

for col in colunas_texto:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.title()
        df[col] = df[col].replace('Nan', pd.NA)

### 5. Tratamento de Missing Values
Exclusão de registros sem data de lançamento ou volume de vendas, pois inviabilizam cálculos de market share e séries temporais. Valores nulos na nota da crítica (`critic_score`) foram mantidos intactos para não enviesar negativamente a volumetria total de vendas de títulos clássicos.

In [ ]:
df = df.dropna(subset=['release_date', 'total_sales'])

### 6. Quality Assurance (QA)
Inspeção final da estrutura do dataframe para confirmar a eficácia do casting, a limpeza de strings e a volumetria dos dados após o descarte das linhas nulas.

In [ ]:
display(df.info())

### 7. Load: Exportação da Base Refinada
Persistência dos dados processados em um novo arquivo `.csv`, preservando o dado bruto original. O índice gerado pelo Pandas é omitido para garantir uma ingestão limpa pela aplicação de visualização gráfica.

In [31]:
df.to_csv(caminho_tratada, index=False)